# Lab 5: extraction as a tool call

Lab 5 Costs: $0.017

Scenario 6, structured data extraction. Northwind's suppliers send invoices, receipts and
credit notes as whatever the sending system happened to produce: a letterhead, a till slip,
a scanned note with the reasons typed underneath. Downstream, a ledger wants the same
handful of fields off every one of them.

This lab builds that one step and nothing else. One document in, one schema-valid record
out, using the Messages API and a tool whose input schema is the record you want back. No
vendor lookup, no validation loop, no retry, no batch. Those are real parts of a real
pipeline and the course teaches them; none of them is what makes the extraction work.

Two halves, in order. First the mechanism: why a prompt asking for JSON is not a contract,
and what a forced tool call gives you instead. Then the schema design, which is where the
interesting failure lives: a field your document does not print, and what Claude does about
it.

This lab calls the API. Every call below is a single turn over a document of a dozen lines.

Setup, once, in the `code/` directory above this one: copy `.env.example` to `.env` and put a
Console key in `ANTHROPIC_API_KEY`. Every lab reads that same file, and this is the one lab
that needs a key rather than a subscription login: it calls the Claude API directly, and a
subscription authenticates the `claude` binary rather than the API.

## 1. Three documents to read

The three documents below are the fixtures for the whole lab, and each one is here to break
something specific later. The invoice is the well-behaved case. The receipt prints no
purchase order and no date, which section 5 needs. The credit note gives two reasons, one
real but off any sensible list and one genuinely ambiguous, which section 6 needs.

They are written into `workspace/` rather than committed, so the cell that creates them is
the cell you read.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

LAB = Path.cwd()
CODE = LAB.parent
if not (CODE / "pyproject.toml").exists():
    raise SystemExit(
        f"Run this notebook from its own folder inside labs/code. The working "
        f"directory is {LAB}, and {CODE / 'pyproject.toml'} is not there. "
        f"In JupyterLab the working directory follows the notebook, so open it "
        f"from the file browser rather than starting the kernel elsewhere."
    )

ENV_FILE = CODE / ".env"
if not ENV_FILE.exists():
    raise SystemExit(f"No {ENV_FILE}. Copy .env.example to .env and read the notes at the top of it.")

load_dotenv(ENV_FILE)

# The one lab of the six that insists on a key. It calls the Claude API directly rather than
# through the Agent SDK, and that client authenticates with a key: a subscription login is a
# credential for the claude binary, not for the API, so it cannot stand in here.
missing = [name for name in ("ANTHROPIC_API_KEY", "LAB_MODEL") if not os.environ.get(name)]
if missing:
    raise SystemExit(
        f"{ENV_FILE} has no value for: {', '.join(missing)}. This lab calls the Claude API "
        f"directly, so a subscription login will not carry it: it needs a Console key."
    )
MODEL = os.environ["LAB_MODEL"]

DOCS = LAB / "workspace" / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

INVOICE = '''
NORTHWIND TIMBER SUPPLIES LTD
Unit 7, Brackley Trading Estate

INVOICE

Invoice number : INV-4471
Invoice date   : 14 March
Your order ref : PO-99812

Description                    Qty      Price
Oak board, 2.4m                 12     540.00
Pine batten, 3m                 40     180.00
Delivery                         1      35.00

                         Total due     755.00

Payment terms: 30 days net
'''

# No purchase order anywhere on it, and no date. Section 5 turns on both absences.
RECEIPT = '''
BRACKLEY BUILDERS MERCHANT
Till 3        Receipt 8802

2 x sealant cartridge           11.80
1 x brush set                    6.40
1 x dust sheet                   4.95

TOTAL                           23.15
CARD ****4417   APPROVED

Thank you for your custom
'''

# Line 1's reason is real and outside any sensible list. Line 2's is ambiguous.
CREDIT_NOTE = '''
NORTHWIND TIMBER SUPPLIES LTD

CREDIT NOTE CN-221
Raised against invoice INV-4471

Line 1   Oak board, 2.4m   x2        -90.00
         Reason: goodwill gesture, the delivery ran three days late

Line 2   Pine batten, 3m   x4        -18.00
         Reason: customer rang, could not say what was wrong with them

                    Total credited  -108.00
'''

for name, body in (("invoice-4471.txt", INVOICE),
                   ("receipt-8802.txt", RECEIPT),
                   ("credit-note-221.txt", CREDIT_NOTE)):
    (DOCS / name).write_text(body.lstrip("\n"), encoding="utf-8")

invoice = (DOCS / "invoice-4471.txt").read_text(encoding="utf-8")
receipt = (DOCS / "receipt-8802.txt").read_text(encoding="utf-8")
credit_note = (DOCS / "credit-note-221.txt").read_text(encoding="utf-8")

print(f"documents in {DOCS}")
print(f"model        from LAB_MODEL in {ENV_FILE}")
print()
print(invoice)

## 2. The failure a schema prevents

Start with the version everybody writes first: ask for JSON in the prompt and parse what
comes back. It works often enough to reach production and fail there.

Watch what the next cell prints before it parses. What arrives is a good reply. It is just
not a record: there is a code fence around it, or a sentence introducing it, or an offer to
break the line items out separately. Any one of those, and `json.loads` raises on character
one.

The problem is not this particular reply. It is that a prompt cannot promise anything about
the next one: not that it parses, not that it carries the same fields with the same types,
not that an absent value comes back empty rather than filled.

In [ ]:
import anthropic

client = anthropic.Anthropic()

asking_nicely = f'''Extract the vendor, the invoice number and the total from this
document. Reply with JSON only.

<document>
{invoice}
</document>'''

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": asking_nicely}],
)

text = next(b.text for b in response.content if b.type == "text")
print(f"stop_reason: {response.stop_reason}")
print("-" * 68)
print(text)
print("-" * 68)

try:
    record = json.loads(text)
    print(f"parsed: {record}")
except json.JSONDecodeError as exc:
    print(f"json.loads failed: {exc}")

## 3. The extraction tool

Now the mechanism the exam calls the most reliable one. It starts with an inversion worth
saying out loud, because the rest follows from it: you define a tool, and what Claude calls
that tool's **input** is what you are treating as your **output**. There is no function
named `extract_invoice` anywhere in this notebook. Nothing runs.

Three parts do the work:

| Part | Job |
|---|---|
| `input_schema` | The record you want back, written as the tool's arguments |
| `tool_choice` | Forces the call, so Claude cannot answer in prose instead |
| `tool_use.input` | Already a dict. Read the record straight off it |

Two details in the next cell are easy to skim past and both matter. The block is found by
its **type**, not its position: a forced call usually puts it first, and usually is not a
contract. And the loop deliberately does not close. `stop_reason` comes back as `tool_use`,
which everywhere else in this course means send a `tool_result` back. Here you already have
what you wanted, so you stop.

In [ ]:
extract_invoice = {
    "name": "extract_invoice",
    "description": "Extracts the header fields from a supplier invoice.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor": {
                "type": "string",
                "description": "The supplier's name exactly as printed at the top.",
            },
            "invoice_number": {
                "type": "string",
                "description": "The supplier's own reference for this invoice.",
            },
            "stated_total": {
                "type": "number",
                "description": "The total the document states, as a number.",
            },
        },
        "required": ["vendor", "invoice_number", "stated_total"],
    },
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": f"Extract the invoice.\n\n{invoice}"}],
    tools=[extract_invoice],
    tool_choice={"type": "tool", "name": "extract_invoice"},
)

# By type, not by position. A thinking block or a line of text can come first.
block = next(b for b in response.content if b.type == "tool_use")

print(f"stop_reason : {response.stop_reason}")
print(f"tool called : {block.name}")
print(f"record      : {block.input}")
print()
print("No tool_result is sent back. The record is the answer, so the loop stops here.")

## 4. The model you validate is the schema you send

The schema above is a dict written by hand, and the record that came back is a dict too. So
nothing yet stops a typo in the schema, and nothing turns the reply into something your
editor understands.

A Pydantic model does both jobs from one definition. `model_json_schema()` generates the
schema you send, and `model_validate()` checks the record you got and hands back a typed
object. One source of truth, so the two cannot drift apart.

`Invoice` below stays flat on purpose. A nested model would generate `$defs` and `$ref`,
which is a JSON Schema question rather than an extraction one; `line_items` is a list of
strings so the schema can show an array without it.

In [ ]:
from pydantic import BaseModel, Field

class Invoice(BaseModel):
    vendor: str = Field(description="The supplier's name exactly as printed at the top.")
    invoice_number: str = Field(description="The supplier's own reference for this invoice.")
    stated_total: float = Field(description="The total the document states, as a number.")
    line_items: list[str] = Field(description="One entry per description line charged for.")

extract_invoice = {
    "name": "extract_invoice",
    "description": "Extracts the header fields from a supplier invoice.",
    "input_schema": Invoice.model_json_schema(),
}

print("the schema the model is given:")
print(json.dumps(extract_invoice["input_schema"], indent=2))

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": f"Extract the invoice.\n\n{invoice}"}],
    tools=[extract_invoice],
    tool_choice={"type": "tool", "name": "extract_invoice"},
)
block = next(b for b in response.content if b.type == "tool_use")

record = Invoice.model_validate(block.input)
print()
print(f"typed record : {record!r}")
print(f"one field    : {record.stated_total + 0}")

## 5. Designing for what the document does not print

Everything so far ran against the invoice, which prints every field the schema asked for.
Now point the same shape of tool at the receipt, which prints no purchase order and no date
at all.

The next cell asks for both as plain required strings. The schema gives Claude no way to
report that the document is silent, so it has to put a string in each field, and it does.

Measured across twenty runs on the two models named in `.env.example`: every single one put
the sentinel `<UNKNOWN>` in the date, and most put it in the purchase order as well. Reword
the description slightly and a smaller model instead lifts the till receipt number into the
purchase order: a real value, off the page, in the wrong field. Which of the two you get is
a property of the model on the day. The shape of the defect is not. Your ledger is handed a
string, it type-checks, the key is required so it is always present, and nothing downstream
can tell it apart from a purchase order the supplier really printed.

So the cell prints the record rather than asserting anything about it. What lands in those
two fields is Claude's to decide; this notebook's job is to show you that it decided.

In [ ]:
strict_receipt = {
    "name": "extract_receipt",
    "description": "Extracts the header fields from a supplier receipt.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor": {"type": "string", "description": "The shop's name as printed."},
            "stated_total": {"type": "number", "description": "The total, as a number."},
            "purchase_order": {
                "type": "string",
                "description": "The purchase order this receipt is against.",
            },
            "issued_on": {
                "type": "string",
                "description": "The date printed on the receipt.",
            },
        },
        "required": ["vendor", "stated_total", "purchase_order", "issued_on"],
    },
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": f"Extract the receipt.\n\n{receipt}"}],
    tools=[strict_receipt],
    tool_choice={"type": "tool", "name": "extract_receipt"},
)
block = next(b for b in response.content if b.type == "tool_use")

print("required, and not nullable:")
for key, value in block.input.items():
    print(f"  {key:16} {value!r}")
print()
print("The receipt prints neither a purchase order nor a date. Check the two above.")

In [ ]:
nullable_receipt = json.loads(json.dumps(strict_receipt))   # copy, so the contrast survives
properties = nullable_receipt["input_schema"]["properties"]
properties["purchase_order"] = {
    "type": ["string", "null"],
    "description": "The purchase order this receipt is against. "
                   "Null when the document does not print one.",
}
properties["issued_on"] = {
    "type": ["string", "null"],
    "description": "The date printed on the receipt. "
                   "Null when the document does not print one.",
}
nullable_receipt["name"] = "extract_receipt_nullable"

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": f"Extract the receipt.\n\n{receipt}"}],
    tools=[nullable_receipt],
    tool_choice={"type": "tool", "name": "extract_receipt_nullable"},
)
block = next(b for b in response.content if b.type == "tool_use")

print("nullable, and still required:")
for key, value in block.input.items():
    print(f"  {key:16} {value!r}")
print()
print("Both keys are still required, so the ledger always finds them. What changed is that")
print("absence now has a value it can be reported as.")

## 6. When the category is not on your list

The last failure is the same one wearing different clothes. A closed list of categories is
a schema demanding a string, so a reason outside the list gets forced into the nearest
member, confidently, and the record looks clean.

Two escape values fix it, and they are not the same escape:

| Value | Means | What you do about it |
|---|---|---|
| `unclear` | The source is ambiguous | Send it to a reviewer |
| `other` | The category is real, your list is short | Consider adding it to the list |

Both put the specifics in a companion detail field, so nothing is thrown away. The credit
note has one of each: a goodwill gesture, which is a real reason no sensible list carries,
and a line where the customer could not say what was wrong.

In [ ]:
extract_credit_note = {
    "name": "extract_credit_note",
    "description": "Extracts the credited lines from a supplier credit note.",
    "input_schema": {
        "type": "object",
        "properties": {
            "credit_note_number": {"type": "string", "description": "The note's reference."},
            "lines": {
                "type": "array",
                "description": "One entry per credited line.",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string", "description": "The goods credited."},
                        "amount": {"type": "number", "description": "The amount, as a number."},
                        "reason": {
                            "type": "string",
                            "enum": ["damaged", "wrong_item", "overcharge",
                                     "short_delivery", "unclear", "other"],
                            "description": "Use unclear when the note is ambiguous about "
                                           "why. Use other when it gives a real reason this "
                                           "list does not carry.",
                        },
                        "reason_detail": {
                            "type": ["string", "null"],
                            "description": "The reason as printed. Required when reason is "
                                           "unclear or other. Null otherwise.",
                        },
                    },
                    "required": ["description", "amount", "reason", "reason_detail"],
                },
            },
        },
        "required": ["credit_note_number", "lines"],
    },
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": f"Extract the credit note.\n\n{credit_note}"}],
    tools=[extract_credit_note],
    tool_choice={"type": "tool", "name": "extract_credit_note"},
)
block = next(b for b in response.content if b.type == "tool_use")

print(f"credit note {block.input['credit_note_number']}")
for line in block.input["lines"]:
    print(f"  {line['amount']:>8.2f}  {line['reason']:<14} {line['reason_detail']!r}")
print()
print("Neither line was forced into damaged or wrong_item, and neither reason was lost.")

## What you built

Read this back against the diagram.

| Diagram box | Where it was built |
|---|---|
| Supplier documents | Section 1 |
| `extract_metadata`, forced tool choice | Sections 3 and 4 |
| JSON Schema | Sections 3, 5 and 6 |
| Pydantic validation | Section 4 |

The diagram carries more boxes than this lab does, on purpose. Vendor lookup through a tool
loop, semantic validation, the repair loop, field-level confidence and review routing, and
the whole Message Batches lane are taught in Chapters 16, 17 and 19 and are not practised
here. This lab is the step they all sit on top of.

The decision to carry out of it: given a field your downstream system always reads and a
document that may not print it, make the type nullable and keep the key required. Nullable
is about the type, required is about the key, and they are not alternatives. That pairing is
what turns an absence into a value your code can act on instead of a plausible invention
nobody queries.